In [1]:
import json
import shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
# Anchor paths from notebook
BASE_DIR = Path().cwd().parents[1]
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed_data"
ML_DATA_DIR = BASE_DIR / "data" / "ml_data"

ROOMS_TO_PROCESS = ["bedrooms", "living_rooms"]

# Allow tracking which sources are allowed per room
ALLOWED_SOURCES = {
    "bedrooms": ["deepfurn", "sklad_mebliv"],
    "living_rooms": ["deepfurn"],
}

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet50.eval()

embedding_model = nn.Sequential(*list(resnet50.children())[:-1])
embedding_model = embedding_model.to(device)
embedding_model.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

device: cpu


In [4]:
def extract_embedding(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = preprocess(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            embedding = embedding_model(image_tensor)
        
        # Flatten to 1D vector (2048-dim)
        embedding = embedding.squeeze().cpu().numpy()
        return embedding
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

In [5]:
for room in ROOMS_TO_PROCESS:
    room_dir = PROCESSED_DATA_DIR / room
    if not room_dir.is_dir():
        continue
        
    print(f"Room: {room}")
    
    furniture_embeddings = {}
    
    for source_dir in sorted(room_dir.iterdir()):
        if not source_dir.is_dir() or source_dir.name.startswith('.'):
            continue
            
        source_name = source_dir.name
        if source_name not in ALLOWED_SOURCES.get(room, []):
            print(f"Skipping source {source_name} for room {room}")
            continue
            
        furniture_embeddings[source_name] = {}
        
        print(f"Processing source: {source_name}")
        
        for scene_dir in tqdm(sorted(source_dir.iterdir()), desc=source_name):
            if not scene_dir.is_dir():
                continue
                
            scene_name = scene_dir.name
            annotation_path = scene_dir / "annotation.json"
            if not annotation_path.exists():
                annotation_path = scene_dir / "annotations.json" # fallback
            
            furnitures_dir = scene_dir / "furnitures"
            
            if not annotation_path.exists() or not furnitures_dir.exists():
                continue
                
            with open(annotation_path, 'r', encoding='utf-8') as f:
                annotation = json.load(f)
                
            furniture_embeddings[source_name][scene_name] = {}
            
            for furniture in annotation.get("furnitures", []):
                furniture_id = furniture.get("furniture_id", None)
                category = furniture.get("category", "Unknown")
                
                furniture_image_path = furnitures_dir / f"{furniture_id}.jpg"
                
                if furniture_image_path.exists():
                    embedding = extract_embedding(furniture_image_path)
                    if embedding is not None:
                        furniture_embeddings[source_name][scene_name][furniture_id] = {
                            "scene_name": scene_name,
                            "category": category,
                            "source": source_name,
                            "embedding": embedding.tolist()
                        }
    
    # Save embeddings for this room
    room_ml_data_dir = ML_DATA_DIR / room / "embeddings"
    room_ml_data_dir.mkdir(parents=True, exist_ok=True)
    embeddings_file = room_ml_data_dir / "furniture_embeddings.json"
    
    with open(embeddings_file, 'w', encoding='utf-8') as f:
        json.dump(furniture_embeddings, f, indent=2)
        
    print(f"Embeddings for {room} saved to {embeddings_file}")

Room: bedrooms
Processing source: deepfurn


deepfurn:   0%|          | 0/807 [00:00<?, ?it/s]

deepfurn: 100%|██████████| 807/807 [05:45<00:00,  2.34it/s]


Skipping source deepfurn_combined for room bedrooms
Processing source: sklad_mebliv


sklad_mebliv: 100%|██████████| 74/74 [00:25<00:00,  2.86it/s]


Skipping source wayfair for room bedrooms
Embeddings for bedrooms saved to /teamspace/studios/this_studio/full_new_thesis_v2/data/ml_data/bedrooms/embeddings/furniture_embeddings.json
Room: living_rooms
Processing source: deepfurn


deepfurn: 100%|██████████| 801/801 [05:50<00:00,  2.29it/s]


Embeddings for living_rooms saved to /teamspace/studios/this_studio/full_new_thesis_v2/data/ml_data/living_rooms/embeddings/furniture_embeddings.json
